<a href="https://colab.research.google.com/github/vimesh630/air-quality-measuring-edge-AI/blob/feature%2Faqi-classifier/train_model_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             accuracy_score)
import tensorflow as tf
import pickle

In [ ]:
# TFLite import — works on Windows (TF fallback) and Raspberry Pi (tflite_runtime)
try:
    import tflite_runtime.interpreter as tflite
except ModuleNotFoundError:
    try:
        from tensorflow.lite.python.interpreter import Interpreter as _Interp
    except ImportError:
        _Interp = tf.lite.Interpreter
    class tflite:
        Interpreter = _Interp

print("=" * 50)
print("AQM Model Training — Day 2")
print("=" * 50)

In [ ]:
# STEP 1 — Load clean dataset
df = pd.read_csv('model/data/aqm_clean.csv')

print(f"\nDataset loaded")
print(f"Rows    : {len(df):,}")
print(f"Columns : {list(df.columns)}")
print(f"\nLabel counts:")
print(df['label'].value_counts())

In [ ]:
# STEP 2 — Encode labels + train/test split
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])

print(f"\nLabel encoding:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} = {cls}")

X = df[['temperature', 'humidity', 'aqi']].values
y = df['label_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain set : {len(X_train):,} rows")
print(f"Test set  : {len(X_test):,} rows")
print(f"\nTrain label distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {le.classes_[u]:<12} {c:>8,} ({c/len(y_train)*100:.1f}%)")


In [ ]:
# STEP 3 — Train Random Forest classifier
print(f"\nTraining Random Forest...")
print(f"(This takes about 30–60 seconds)")

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print("Training complete.")

y_pred   = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n── Results ─────────────────────────────")
print(f"Overall accuracy : {accuracy * 100:.2f}%")
print(f"\nDetailed report:")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_
))